# ModelCallLimitMiddleware中间件
限制模型调用次数，避免无限循环，控制调用成本。

In [1]:
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from typing import List

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,  # 每个线程最多2次模型调用
            # run_limit=5,  # 每次运行最多5次
            exit_behavior="end",  # 达到限制后退出
        ),
    ],
)


def pretty_iterate_msg(
    messages: List[SystemMessage | HumanMessage | AIMessage | ToolMessage],
):
    for msg in messages:
        msg.pretty_print()


config = {"configurable": {"thread_id": "1"}}

response_first = agent.invoke(
    {"messages": [HumanMessage("你好")]},
    config=config,
)
print("=" * 30, "> first <", "=" * 30)
pretty_iterate_msg(response_first["messages"])

response_second = agent.invoke(
    {"messages": [HumanMessage("你是谁？")]},
    config=config,
)
print("=" * 30, "> second <", "=" * 30)
pretty_iterate_msg(response_second["messages"])

response_third = agent.invoke(
    {"messages": [HumanMessage("你能帮我做什么？")]},
    config=config,
)
print("=" * 30, "> third <", "=" * 30)
pretty_iterate_msg(response_third["messages"])